# BRSET: preprocessing and inference (plan v6)

Runs **only after** `04_results/locked_config_v6_brset.json` exists, and refuses otherwise. It:

1. extracts the verified BRSET archive and checks it against the access record;
2. preprocesses every image with the same pipeline as EyePACS and DDR (RGB, 512x512 Lanczos, PNG);
3. runs 45 frozen models: the 33 EyePACS-trained models, plus the 12 DDR-trained models (DenseNet121, EfficientNet-B3, ResNet50, RETFound; 5-class; 3 runs each);
4. writes one prediction file per model (image id and score only) and hashes each.

**It never opens the label file.** It only checks that the label file is byte-identical to the one recorded at download. No performance metric is computed. The analysis notebook is run once, afterwards.

Data use: credentialed PhysioNet data. Prediction files are record-level and stay in this Drive. Only the aggregate analysis output is shared.

A100, High-RAM; `HF_TOKEN` secret for RETFound. Roughly 1.5 to 2 hours. Resumable: models whose prediction file exists are skipped.

In [5]:
# Cell 1: environment and guards
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install timm transformers
import os, glob, json, hashlib, datetime, tarfile, time
import numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
CKPT, PREDS, RESULTS, AUDIT = (f'{BASE}/{d}' for d in ('02_checkpoints', '03_predictions', '04_results', '99_audit'))
BR = '/content/drive/MyDrive/stochastic_orthogonality/data/brset'
LOCK_FILE = f'{RESULTS}/locked_config_v6_brset.json'
def sha256_file(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
assert os.path.exists(LOCK_FILE), 'BRSET plan (lock v6) missing: do not run.'
LOCK_SHA = sha256_file(LOCK_FILE)
ACC = json.load(open(f'{BR}/brset_access_record.json'))
assert sha256_file(f'{BR}/label_brset.csv') == ACC['label_file_sha256'], 'Label file differs from the one recorded at download.'
existing = glob.glob(f'{PREDS}/brset_predictions_*_v2.csv')
if existing: assert os.path.getmtime(LOCK_FILE) < min(os.path.getmtime(p) for p in existing), 'A BRSET prediction predates the lock.'
EYEPACS = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class','resnet50_binary','resnet50_5class',
           'vit_base_binary','vit_base_5class','retfound_binary','retfound_5class','retfound_adversarial']
DDR = ['densenet121_5class','efficientnet_b3_5class','resnet50_5class','retfound_5class']
ROSTER = [('eyepacs', m, k) for k in (0, 1, 2) for m in EYEPACS] + [('ddr', m, k) for k in (0, 1, 2) for m in DDR]
def tag(k): return '' if k == 0 else f'_seed{k}'
for src, m, k in ROSTER: assert os.path.exists(f'{CKPT}/{src}_{m}{tag(k)}_v2.pt'), f'missing checkpoint {src}_{m}{tag(k)}'
print('Lock v6', LOCK_SHA[:16], '| access record OK | 45 checkpoints present | GPU:', torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Lock v6 b8d4c97a1167ba77 | access record OK | 45 checkpoints present | GPU: NVIDIA A100-SXM4-40GB


In [6]:
# Cell 2: extract the verified archive and preprocess (same pipeline as EyePACS and DDR)
LOCAL = '/content/brset_local'; PNG = '/content/brset_png'
for d in (LOCAL, PNG): os.makedirs(d, exist_ok=True)
TAR = f'{BR}/{ACC["archive"]}'
if not glob.glob(f'{LOCAL}/**/fundus_photos', recursive=True):
    assert sha256_file(TAR) == ACC['archive_sha256'], 'Archive differs from the access record.'
    with tarfile.open(TAR) as t: t.extractall(LOCAL)
photos = sorted(glob.glob(f'{LOCAL}/**/fundus_photos/*', recursive=True))
IDS = [os.path.splitext(os.path.basename(p))[0] for p in photos]
assert len(IDS) == ACC['image_files'] and len(set(IDS)) == len(IDS), (len(IDS), ACC['image_files'])
from multiprocessing import Pool
def v1_preprocess(fp): return Image.open(fp).convert('RGB').resize((512, 512), Image.LANCZOS)
def _work(fp):
    out = f'{PNG}/{os.path.splitext(os.path.basename(fp))[0]}.png'
    if os.path.exists(out) and os.path.getsize(out) > 0: return 'skip'
    try: v1_preprocess(fp).save(out); return 'ok'
    except Exception as e: return f'error: {e}'
t0 = time.time()
with Pool(os.cpu_count()) as pool: res = list(pool.imap_unordered(_work, photos, chunksize=32))
errs = [r for r in res if r.startswith('error')]; assert not errs, errs[:5]
print(f'{len(IDS)} images preprocessed in {time.time() - t0:.0f}s')

/tmp/ipykernel_2720/1876684767.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(TAR) as t: t.extractall(LOCAL)


16266 images preprocessed in 522s


In [7]:
# Cell 3: frozen models, strict loading, prediction (no labels, no metrics)
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoConfig, AutoModel
RETFOUND_REPO = 'iszt/RETFound_mae_meh'
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                              transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
class ImgDS(Dataset):
    def __init__(self, ids): self.ids = list(ids)
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return TF_EVAL(Image.open(f'{PNG}/{self.ids[i]}.png').convert('RGB')), i
class RETFound(nn.Module):
    def __init__(self, nc):
        super().__init__(); cfg = AutoConfig.from_pretrained(RETFOUND_REPO)
        self.base = AutoModel.from_config(cfg, add_pooling_layer=False)
        self.fc_norm = nn.LayerNorm(cfg.hidden_size, eps=1e-6); self.classifier = nn.Linear(cfg.hidden_size, nc)
    def forward(self, x):
        h = self.base(pixel_values=x).last_hidden_state
        return self.classifier(self.fc_norm(h[:, 1:].mean(dim=1)))
def load_model(src, name, k, dev):
    path = f'{CKPT}/{src}_{name}{tag(k)}_v2.pt'
    blob = torch.load(path, map_location='cpu', weights_only=False); cfg = blob['config']
    m = RETFound(cfg['nc']) if cfg['arch'] == 'retfound' else timm.create_model(cfg['arch'], pretrained=False, num_classes=cfg['nc'])
    m.load_state_dict(blob['state_dict'], strict=True)
    return m.to(dev).eval(), cfg, sha256_file(path)
@torch.no_grad()
def predict(model, dev, bs):
    probs = np.zeros(len(IDS), dtype=np.float64)
    for x, idx in DataLoader(ImgDS(IDS), batch_size=bs, shuffle=False, num_workers=min(8, os.cpu_count()), pin_memory=True):
        out = model(x.to(dev, non_blocking=True)).float()
        p = torch.sigmoid(out[:, 0]) if out.shape[1] == 1 else torch.softmax(out, 1)[:, 2:].sum(1)
        probs[idx.numpy()] = p.double().cpu().numpy()
    return probs
dev = 'cuda'; RECORD = {}
todo = [(s, m, k) for s, m, k in ROSTER if not os.path.exists(f'{PREDS}/brset_predictions_{s}_{m}{tag(k)}_v2.csv')]
print(f'to run: {len(todo)} of {len(ROSTER)}')
for i, (s, m, k) in enumerate(todo, 1):
    t0 = time.time(); model, cfg, ck_sha = load_model(s, m, k, dev)
    pr = predict(model, dev, 32 if cfg['arch'] == 'retfound' else 128)
    out = f'{PREDS}/brset_predictions_{s}_{m}{tag(k)}_v2.csv'
    pd.DataFrame({'image_id': IDS, 'predicted_probability': pr, 'model_name': m, 'training_data': s, 'replicate': k}).to_csv(out, index=False, float_format='%.17g')
    RECORD[f'{s}_{m}{tag(k)}'] = {'checkpoint_sha256': ck_sha, 'prediction_sha256': sha256_file(out)}
    print(f'[{i}/{len(todo)}] {s} {m}{tag(k)}  {time.time() - t0:.0f}s')
    del model; torch.cuda.empty_cache()

to run: 45 of 45
[1/45] eyepacs densenet121_binary  43s
[2/45] eyepacs densenet121_5class  45s
[3/45] eyepacs efficientnet_b3_binary  42s
[4/45] eyepacs efficientnet_b3_5class  42s
[5/45] eyepacs resnet50_binary  45s
[6/45] eyepacs resnet50_5class  45s
[7/45] eyepacs vit_base_binary  52s
[8/45] eyepacs vit_base_5class  49s


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

[9/45] eyepacs retfound_binary  149s
[10/45] eyepacs retfound_5class  147s
[11/45] eyepacs retfound_adversarial  148s
[12/45] eyepacs densenet121_binary_seed1  41s
[13/45] eyepacs densenet121_5class_seed1  41s
[14/45] eyepacs efficientnet_b3_binary_seed1  43s
[15/45] eyepacs efficientnet_b3_5class_seed1  41s
[16/45] eyepacs resnet50_binary_seed1  47s
[17/45] eyepacs resnet50_5class_seed1  46s
[18/45] eyepacs vit_base_binary_seed1  49s
[19/45] eyepacs vit_base_5class_seed1  49s
[20/45] eyepacs retfound_binary_seed1  147s
[21/45] eyepacs retfound_5class_seed1  147s
[22/45] eyepacs retfound_adversarial_seed1  147s
[23/45] eyepacs densenet121_binary_seed2  40s
[24/45] eyepacs densenet121_5class_seed2  39s
[25/45] eyepacs efficientnet_b3_binary_seed2  39s
[26/45] eyepacs efficientnet_b3_5class_seed2  39s
[27/45] eyepacs resnet50_binary_seed2  42s
[28/45] eyepacs resnet50_5class_seed2  43s
[29/45] eyepacs vit_base_binary_seed2  53s
[30/45] eyepacs vit_base_5class_seed2  49s
[31/45] eyepacs r

In [8]:
# Cell 4: provenance record (no metrics)
done = sorted(glob.glob(f'{PREDS}/brset_predictions_*_v2.csv')); assert len(done) == 45, f'{len(done)} of 45 present'
rec = {'written_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'lock_v6_sha256': LOCK_SHA,
       'access_record_label_sha256': ACC['label_file_sha256'], 'images': len(IDS), 'models': len(done),
       'prediction_sha256': {os.path.basename(p): sha256_file(p) for p in done}, 'models_run_this_session': RECORD,
       'note': 'Labels not opened. No BRSET performance metric computed. The analysis notebook is run once, afterwards.'}
with open(f'{AUDIT}/brset_inference_record_v2.json', 'w') as f: json.dump(rec, f, indent=2)
with open(f'{AUDIT}/RUN_LOG.md', 'a') as f:
    f.write(f"\n\n## BRSET inference ({rec['written_utc'][:10]})\n\nLock v6 {LOCK_SHA[:16]}...; {len(IDS)} images; 45 prediction files. Labels not opened; no metrics. Record 99_audit/brset_inference_record_v2.json.\n")
drive.flush_and_unmount(); drive.mount('/content/drive')
print(json.dumps({k: v for k, v in rec.items() if k not in ('prediction_sha256', 'models_run_this_session')}, indent=1))

Mounted at /content/drive
{
 "written_utc": "2026-10-03T03:38:03.988908+00:00",
 "lock_v6_sha256": "b8d4c97a1167ba777e3f41e2fe7d67efcce5725a2b8d63a78770a9ac74ab70a6",
 "access_record_label_sha256": "4cd000ec1f651deb5cfb4f018ed1dec7a874da59a20c47f1e59a421d4a112adf",
 "images": 16266,
 "models": 45,
 "note": "Labels not opened. No BRSET performance metric computed. The analysis notebook is run once, afterwards."
}
